# Time-series EDA: trend, seasonality, autocorrelation, stationarity

_Data Wrangling_

**Look at data indexed by time the right way before you ever model or forecast it.**

A time series is one column whose order is information. The whole craft of time-series EDA is
       to split that single squiggly line into parts you can reason about, and then to check whether what is
       left behaves nicely.

       Decomposition. The classic mental model is that an observed value is the sum (or product) of
       three pieces: a trend $T_t$ (the slow drift), a seasonality $S_t$ (a pattern that repeats
       on a fixed period $m$ &mdash; 7 for weekly, 12 for monthly, 365 for yearly), and a residual
       $R_t$ (everything left over). seasonal_decompose or STL (Seasonal-Trend decomposition using
       Loess) pulls these apart so you can see each on its own.

---

This notebook walks through time-series EDA one move at a time. Run each cell, read the note above it, then tackle the practice problems at the end. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — pandas + statsmodels

We take one daily series and run the standard EDA pipeline on it, in four steps: (1) build the series and plot it, (2) decompose it into trend + seasonality + residual, (3) read drift with rolling stats and structure with the ACF/PACF, (4) test stationarity and resample to a coarser cadence.

### Step 1 — Build a series on a DatetimeIndex and plot it

Every time-series tool assumes evenly spaced, time-ordered values, so the series must sit on a `DatetimeIndex` and be sorted by time. We synthesize a year of daily data with a known recipe — a slow upward trend, a yearly sine, a weekend bump, and noise — so we know exactly what the later steps *should* recover. The raw plot is always the first look.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose, STL
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller

# Build / load a series on a DatetimeIndex.
# Real data: s = pd.read_csv("sales.csv", parse_dates=["date"]).set_index("date")["y"]
rng = np.random.RandomState(42)
idx = pd.date_range("2021-01-01", periods=365, freq="D")     # a DatetimeIndex
t   = np.arange(365)

trend       = 50 + 0.08 * t                       # slow upward trend
seasonal    = 12 * np.sin(2 * np.pi * t / 365.25) # yearly seasonality
weekend     = 6 * (idx.dayofweek >= 5)            # weekend bump
noise       = rng.normal(0, 3, 365)               # irregular noise
y = trend + seasonal + weekend + noise

s = pd.Series(y, index=idx).sort_index()          # always sort by time!

# PLOT the raw series.
s.plot(title="Raw daily series", figsize=(10, 3))
plt.show()

### Step 2 — Decompose into trend + seasonality + residual

The classic mental model is that an observed value is the sum of a trend, a repeating seasonal pattern, and a residual. `seasonal_decompose` splits the series using simple moving averages; `STL` (Seasonal-Trend decomposition using Loess) is a more robust alternative that resists outliers. We use a weekly `period=7` so the seasonal panel captures the weekend bump.

In [ ]:
# DECOMPOSE into trend + seasonality + residual.
dec = seasonal_decompose(s, model="additive", period=7)   # weekly period
dec.plot()                                                # trend / seasonal / resid
plt.show()

stl = STL(s, period=7, robust=True).fit()                 # robust alternative
stl.plot()
plt.show()

### Step 3 — Read drift with rolling stats and structure with ACF/PACF

A 30-day rolling mean exposes the slow drift (trend) while a rolling std reveals whether the spread is changing over time. The autocorrelation function (ACF) measures how each value correlates with its own past; a spike at lag 7 is the signature of weekly seasonality. The partial ACF (PACF) strips out indirect effects to show the *direct* dependence at each lag.

In [ ]:
# ROLLING mean & std -> drift and changing variance.
roll_mean = s.rolling(30).mean()      # 30-day backward moving average (trend)
roll_std  = s.rolling(30).std()       # 30-day rolling spread (variance drift)

ax = s.plot(alpha=.4, label="raw", figsize=(10, 3))
roll_mean.plot(ax=ax, label="30d mean")
roll_std.plot(ax=ax, label="30d std")
ax.legend()
plt.show()

# AUTOCORRELATION: ACF and PACF.
plot_acf(s, lags=30)      # spike at lag 7 -> weekly seasonality
plt.show()
plot_pacf(s, lags=30)     # direct (partial) dependence per lag
plt.show()

### Step 4 — Test stationarity and resample

The Augmented Dickey-Fuller (ADF) test asks whether the series is stationary; its null hypothesis is *non*-stationary, so a small p-value (< 0.05) lets us call it stationary. The raw series has a trend and usually fails, but its first difference typically passes. Finally, resampling to a weekly cadence gives a calmer view, and comparing the full daily date range against the actual index flags any missing days.

In [ ]:
# STATIONARITY: Augmented Dickey-Fuller test.
def adf_report(series, name):
    stat, p, *_ = adfuller(series.dropna())
    verdict = "stationary" if p < 0.05 else "NON-stationary"
    print(f"{name:18s} ADF={stat:7.3f}  p={p:.4f}  -> {verdict}")

adf_report(s,        "level")           # trend present -> expect non-stationary-ish
adf_report(s.diff(), "1st difference")  # removing the trend usually passes

# RESAMPLE to a coarser cadence + spot gaps.
weekly = s.resample("W").mean()         # calmer weekly view
print(weekly.head())

full_range  = pd.date_range(s.index.min(), s.index.max(), freq="D")
missing     = full_range.difference(s.index)
print("missing dates:", missing)        # empty here; non-empty flags gaps

## Visualize the data & results

_On a real-pattern daily series (trend + yearly season + weekend bump + noise), what does a 30-day rolling mean reveal, and is the raw series stationary?_

### Step 1 — Rebuild the series and a 30-day rolling mean

We reconstruct the same daily series and compute its 30-day rolling mean, then print a thinned sample of both. The rolling mean is much smoother than the raw values: averaging 30 days suppresses the weekend bump and noise, leaving the slow trend plus the yearly wave.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.RandomState(42)
idx = pd.date_range("2021-01-01", periods=365, freq="D")   # DatetimeIndex
t   = np.arange(365)

trend    = 50 + 0.08 * t                        # slow upward trend
seasonal = 12 * np.sin(2 * np.pi * t / 365.25)  # yearly seasonality
weekend  = 6 * (idx.dayofweek >= 5)             # weekend bump
noise    = rng.normal(0, 3, 365)                # noise
s = pd.Series(trend + seasonal + weekend + noise, index=idx)

roll = s.rolling(30).mean()                     # 30-day backward moving average

sub = np.arange(29, 365, 6)[:56]                # 56 plotted days, rolling defined
print("raw :", np.round(s.values[sub], 1))
print("roll:", np.round(roll.values[sub], 1))

# Stationarity (ADF) — needs statsmodels.tsa.stattools.adfuller:
#   adfuller(s)            -> stat -6.14  (level, trend present)
#   adfuller(s.diff()...)  -> stat -22.07 (1st difference, clearly stationary)

### Step 2 — Measure autocorrelation by hand

This small `acf` helper computes the autocorrelation at each lag directly: center the series, then for each lag `k` correlate it with a copy of itself shifted by `k`. On the trended series the ACF decays only slowly (a drifting mean keeps values correlated far apart); on a clean weekly series it shows a sharp spike at lag 7, the seasonal period.

In [ ]:
# Autocorrelation of the raw series (slow decay = trend).
def acf(x, nlags):
    x = np.asarray(x, float) - np.mean(x)        # center the series
    denom = (x * x).sum()                        # total variance (lag 0)
    out = []
    for k in range(nlags + 1):
        numer = (x[:len(x) - k] * x[k:]).sum()   # overlap of series with its lag-k shift
        out.append(round(float(numer / denom), 3))
    return out

print("ACF 0..7:", acf(s.values, 7))   # [1.0, 0.70, 0.59, 0.60, 0.56, 0.57, 0.66, 0.75]

# A clean weekly series shows the seasonal ACF spike at lag 7:
w = 20 + 8 * np.sin(2 * np.pi * np.arange(140) / 7) + rng.normal(0, 1.5, 140)
print("weekly ACF lag7:", acf(w, 7)[7])  # ~0.87

### Step 3 — Resample to monthly means

Aggregating to a monthly cadence collapses the daily noise into 12 numbers, making the yearly seasonal shape easy to read — values rise toward mid-year and fall back, tracking the sine wave plus the trend.

In [ ]:
print("monthly:", np.round(s.resample("M").mean().values, 1))
# monthly -> [55.6 63.1 68.9 71.5 71.1 69.2 65.  61.6 60.5 63.7 69.2 76. ]

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** You have two years of daily website sessions. Before forecasting, your colleague suggests a random 80/20 train/test split "like we always do". Why is that wrong here, and what do you do instead?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recognize the data is time-ordered, so order carries information and the rows are not exchangeable. — _Sessions on consecutive days are correlated and the whole point is to predict the future from the past._
- See that a random split puts some future days in the training set and some past days in the test set. — _The model then "sees" the future during training &mdash; classic leakage that inflates measured accuracy._
- Split by time: train on the earlier window, test on the later one; never shuffle. — _An honest forecast evaluation mirrors deployment, where only past data is available when predicting tomorrow._

**Answer:** A random split leaks the future: future days land in training and past days in test, so the model is graded on information it would never have at prediction time. Instead split by time &mdash; train on, say, the first 20 months, test on the last 4 &mdash; and never shuffle. Same rule for cross-validation: use forward-chaining (expanding-window) folds, not random ones.

</details>

**Problem 2.** A daily sales series climbs steadily and its plot_acf shows bars that stay high and decay very slowly across 30 lags, plus a noticeable bump at lag 7. The ADF test gives a large (non-negative) p-value. Read these three signals and prescribe the fix.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Interpret the slowly decaying ACF as the fingerprint of a trend. — _When the mean drifts, every value stays correlated with many past values, so the autocorrelation decays only gradually._
- Interpret the lag-7 bump as weekly seasonality. — _Each day resembles the same weekday last week, producing a spike at the seasonal period $m=7$._
- Read the large ADF p-value as a failure to reject non-stationarity, then difference. — _ADF's null is "has a unit root" (non-stationary); a large p-value means you cannot reject it, so the raw series is non-stationary. diff(1) removes the trend, diff(7) removes the weekly season; re-test after each, and stop once it passes._

**Answer:** The slowly decaying ACF says trend; the lag-7 spike says weekly seasonality; the large ADF p-value confirms the raw series is non-stationary. Fix it by differencing: y.diff(1) to remove the trend and/or y.diff(7) for the weekly season, re-running adfuller after each step. Stop the moment the test passes &mdash; differencing again just over-differences and adds noise.

</details>

**Problem 3.** Your hourly sensor feed has missing hours scattered through it and you want a 24-hour rolling mean and a clean ACF. What goes wrong if you ignore the gaps, and how do you prepare the series?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Notice that rolling windows and lag-$k$ autocorrelation assume evenly spaced timestamps. — _A "24-row" window or a "lag-24" comparison only equals 24 hours if every hour is present; gaps make a window span more real time than intended._
- Reindex onto a complete regular hourly grid so the missing hours become explicit NaN rows. — _series.asfreq("H") (or reindex to a full date_range) exposes the holes instead of silently skipping them._
- Decide a fill policy &mdash; interpolate short gaps, or leave longer ones as NaN and let the window require a minimum count. — _Short gaps interpolate safely; long gaps should not be invented, so you keep them missing and handle them honestly downstream._

**Answer:** If you ignore the gaps, a "24-row" rolling window and a "lag-24" autocorrelation no longer correspond to 24 real hours, so both the smoothed line and the ACF are wrong. Reindex to a regular hourly grid with asfreq("H") so missing hours appear as NaN, then choose a fill policy &mdash; interpolate short gaps, leave long ones missing. Only then do rolling stats and the ACF measure what you think they measure.

</details>